# NANOGrav 20-Year (NG20) Astrophysical Interpretation: Fiducial Parameter Space

This notebook demonstrates the proposed **NG20 Fiducial Parameter Space** (`PS_NG20_Fiducial`) designed for the flagship NANOGrav 20-year astrophysics interpretation analysis.

### Framework Summary
1. **Binary Evolution / Hardening:** Blecha Inside-Out (BIO) Hardening (`FixedOuterTime_InnerPL_SAM`, Model 1: boundary rate $da/dt|_{r_\mathrm{char}}$ with guaranteed physical subluminal speeds).
2. **Galaxy Stellar Mass Function (GSMF):** Double-Schechter parameterization with the full 11-dimensional covariance matrix from **Leja et al. (2020)** (`PD_MVNormal`).
3. **Galaxy Merger Rate (GMR):** Cosmological simulation-derived merger rate from **Illustris / Rodriguez-Gomez et al. (2015)** (`GMR_Illustris`).
4. **$M_\mathrm{BH}$–Host Galaxy Scaling Relation:** **Kormendy & Ho (2013)** $M_\mathrm{BH}$–$M_\mathrm{bulge}$ relation with redshift-evolving normalization $M_\mathrm{amp}(z) = M_\mathrm{amp,0}(1+z)^{\gamma_z}$ (**Matt et al. 2026a**).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import holodeck as holo
from holodeck import librarian
from holodeck.librarian.param_spaces import PS_NG20_Fiducial
from holodeck.librarian.param_spaces_classic import PS_Classic_Phenom_Astro_Extended
from holodeck.constants import MSOL, GYR, PC, YR

print("Holodeck version:", holo.__version__)

## 1. Parameter Space Instantiation & Structure

In [ ]:
# Instantiate the NG20 Fiducial parameter space with N=1000 Latin Hypercube samples
NSAMPLES = 1000
ps_ng20 = PS_NG20_Fiducial(holo.log, nsamples=NSAMPLES, seed=42)

print(f"Total free parameter dimensions: {len(ps_ng20.param_names)}")
print(f"Sample array shape: {ps_ng20.param_samples.shape}\n")

print("Parameter Names and Defaults:")
for name in ps_ng20.param_names:
    def_val = ps_ng20.DEFAULTS.get(name, 'N/A')
    print(f"  - {name:<26}: default = {def_val}")

## 2. Prior Distributions: NG20 Fiducial vs. NG15 Classic Analysis

Here we compare the prior distributions between the **NG20 Fiducial** parameter space and the **NG15 Classic Phenomenological Extended** parameter space (`PS_Classic_Phenom_Astro_Extended`).

In [ ]:
# Sample NG15 parameter space for comparison
ps_ng15 = PS_Classic_Phenom_Astro_Extended(holo.log, nsamples=NSAMPLES, seed=42)

# Extract sample arrays into parameter dictionaries
samples_ng20 = {name: ps_ng20.param_samples[:, i] for i, name in enumerate(ps_ng20.param_names)}
samples_ng15 = {name: ps_ng15.param_samples[:, i] for i, name in enumerate(ps_ng15.param_names)}

# Plot key prior distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=120)
axes = axes.flatten()

# 1. M-Mbulge Normalization (z=0)
ax = axes[0]
ax.hist(samples_ng20['mmb_mamp_log10'], bins=25, density=True, alpha=0.6, label='NG20 Fiducial (KH2013)', color='tab:blue')
ax.hist(samples_ng15['mmb_mamp_log10'], bins=25, density=True, alpha=0.6, label='NG15 Classic', color='tab:orange')
ax.set_xlabel(r'$\log_{10}(M_{\rm amp,0} / M_\odot)$ at $z=0$')
ax.set_ylabel('Probability Density')
ax.set_title(r'$M_{\rm BH}$–$M_{\rm bulge}$ Normalization')
ax.legend(fontsize=8)

# 2. M-Mbulge Slope
ax = axes[1]
ax.hist(samples_ng20['mmb_plaw'], bins=25, density=True, alpha=0.6, label='NG20 Fiducial', color='tab:blue')
ax.hist(samples_ng15['mmb_plaw'], bins=25, density=True, alpha=0.6, label='NG15 Classic', color='tab:orange')
ax.set_xlabel(r'Slope $\alpha_{\rm MMB}$')
ax.set_ylabel('Probability Density')
ax.set_title(r'$M_{\rm BH}$–$M_{\rm bulge}$ Slope')
ax.legend(fontsize=8)

# 3. M-Mbulge Redshift Evolution (Matt et al. 2026a)
ax = axes[2]
ax.hist(samples_ng20['mmb_zplaw_amp'], bins=25, density=True, alpha=0.6, label='NG20 Fiducial (Matt+2026a)', color='tab:blue')
ax.axvline(0.0, color='tab:orange', linestyle='--', linewidth=2, label=r'NG15 (Fixed $\gamma_z=0$)')
ax.set_xlabel(r'Redshift Power-Law $\gamma_z$ [$(1+z)^{\gamma_z}$]')
ax.set_ylabel('Probability Density')
ax.set_title(r'$M_{\rm BH}$ Normalization Redshift Evolution')
ax.legend(fontsize=8)

# 4. Outer Hardening Time Delay
ax = axes[3]
ax.hist(samples_ng20['hard_outer_time'], bins=25, density=True, alpha=0.6, label='NG20 (BIO Outer Delay)', color='tab:blue')
ax.hist(samples_ng15['hard_time'], bins=25, density=True, alpha=0.6, label='NG15 (Total Phenom Time)', color='tab:orange')
ax.set_xlabel(r'Time Delay [Gyr]')
ax.set_ylabel('Probability Density')
ax.set_title('Outer / Galaxy Hardening Delay')
ax.legend(fontsize=8)

# 5. BIO Hardening Characteristic Radius r_char
ax = axes[4]
ax.hist(samples_ng20['hard_rchar_9'], bins=np.logspace(-1, 1, 25), density=True, alpha=0.6, label=r'NG20 BIO $r_{\rm char,9}$', color='tab:blue')
ax.set_xscale('log')
ax.set_xlabel(r'Characteristic Radius $r_{\rm char,9}$ [pc]')
ax.set_ylabel('Probability Density')
ax.set_title(r'BIO Transition Radius ($10^9\,M_\odot$)')
ax.legend(fontsize=8)

# 6. BIO Hardening Boundary Speed
ax = axes[5]
ax.hist(samples_ng20['hard_log10_dadt_rchar'], bins=25, density=True, alpha=0.6, label=r'NG20 $\log_{10}|da/dt|_{r\rm char}$', color='tab:blue')
ax.axvline(np.log10(3e10), color='tab:red', linestyle=':', label='Speed of Light ($c$)')
ax.set_xlabel(r'$\log_{10}|da/dt|_{r\rm char}$ [cm/s]')
ax.set_ylabel('Probability Density')
ax.set_title('Inner Hardening Boundary Speed (Safe)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 2.2 Leja+2020 GSMF Covariance Structure

In [ ]:
# Plot correlations among Leja+2020 GSMF parameters
fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
scatter = ax.scatter(
    samples_ng20['gsmf_log10_mstar_z0'],
    samples_ng20['gsmf_log10_phi_one_z0'],
    c=samples_ng20['gsmf_alpha_two'],
    cmap='viridis',
    alpha=0.6,
    s=15
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label(r'Low-mass Slope $\alpha_2$')
ax.set_xlabel(r'Characteristic Mass $\log_{10}(M_*) $ at $z=0$')
ax.set_ylabel(r'Normalization $\log_{10}(\Phi_1)$ at $z=0$')
ax.set_title('Leja et al. (2020) Covariant GSMF Joint Distribution (LHS Samples)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Model Generation & Fast Spectrum Verification

Below we generate the Semi-Analytic Model (`Semi_Analytic_Model`) and BIO-Hardening (`FixedOuterTime_InnerPL_SAM`) objects from sample parameter dictionaries and compute GWB spectra on a compact grid to verify functionality.

In [ ]:
# Frequency bins for 16-yr baseline
freqs, f_edges = holo.utils.pta_freqs(dur=16.03*YR, num=40)

# Construct default model on full production grid shape=[100, 100, 100]
sam_def, hard_def = ps_ng20.model_for_params(ps_ng20.DEFAULTS, sam_shape=[100, 100, 100])
print("Constructed Default SAM:", sam_def)
print("Constructed Default Hardening:", hard_def)

# Calculate GWB for default parameters (2 realizations)
h_ss_def, h_bg_def = sam_def.gwb(f_edges, hard=hard_def, realize=2)

# Plot resulting default spectrum
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
f_yr = freqs * YR

ax.plot(f_yr, h_bg_def[:, 0], label='Fiducial Default Realization 1', color='navy', lw=2)
ax.plot(f_yr, h_bg_def[:, 1], label='Fiducial Default Realization 2', color='royalblue', lw=2, linestyle='--')

# Standard f^(-2/3) power law reference line
f_ref = f_yr[5]
h_ref = h_bg_def[5, 0]
ax.plot(f_yr, h_ref * (f_yr / f_ref)**(-2/3), color='gray', linestyle=':', label=r'$f^{-2/3}$ reference')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Observed GW Frequency [1/yr]')
ax.set_ylabel('Characteristic Strain $h_c(f)$')
ax.set_title('NG20 Fiducial Baseline GWB Spectrum')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 4. Instructions for High-Resolution Library Generation

To produce full library datasets with high resolution (`sam_shape=[100, 100, 100]` or higher) for MCMC model evaluation:

```bash
# Generate a full library with 10,000 parameter points using gen_lib_sams.py
python scripts/gen_lib_sams.py --pspace PS_NG20_Fiducial --nsamples 10000 --shape 100 100 100 --outdir ./output/ng20_fiducial_lib/
```

Or via the SLURM orchestration script:
```bash
sbatch scripts/run_holodeck_lib_gen.sh PS_NG20_Fiducial 10000
```